In [ ]:
from pathlib import Path
import xarray as xr
import matplotlib.pyplot as plt

In [ ]:
# Project root (one level above notebooks/)
BASE = Path.cwd().parent

print("Project directory:")
print(BASE)

INPUT = (
    BASE
    / "data"
    / "corddex-aus-22"
    / "MPI-M-MPI-ESM-LR_CCLM5-0-15"
    / "tasmax_VAN_sim-hist-fut_3km_BC_MPI-M-MPI-ESM-LR_r1i1p1_CLMcom-HZG-CCLM5-0-15_v1_day_1950-2100_leaf-year_rcp26.nc"
)

OUTDIR = BASE / "output"
OUTDIR.mkdir(exist_ok=True)

TXX_DIR = OUTDIR / "txx"
TXX_DIR.mkdir(exist_ok=True)

FIG_DIR = OUTDIR / "figures"
FIG_DIR.mkdir(exist_ok=True)

print(INPUT)

In [ ]:
print("Reading dataset...")

ds = xr.open_dataset(INPUT)

ds

In [ ]:
if "tasmax" in ds.data_vars:
    var = "tasmax"
else:
    var = list(ds.data_vars)[0]

print("Variable =", var)
print("Units =", ds[var].attrs.get("units", "Unknown"))

In [ ]:
print("Computing annual TXx...")

txx = ds[var].groupby("time.year").max(dim="time")

txx.name = "txx"

txx.attrs["long_name"] = "Annual Maximum of Daily Maximum Temperature"
txx.attrs["units"] = ds[var].attrs.get("units", "")

In [ ]:
if ds[var].attrs.get("units", "").lower() in ["k", "kelvin"]:
    txx = txx - 273.15
    txx.attrs["units"] = "degC"

print("Units:", txx.attrs["units"])

In [ ]:
outfile = TXX_DIR / "TXx_annual_1950_2100.nc"

txx.to_netcdf(outfile)

print("Saved to:")
print(outfile)

In [ ]:
first_year = int(txx.year.values[0])

plt.figure(figsize=(8,6))

txx.sel(year=first_year).plot(
    cmap="hot",
    robust=True
)

plt.title(f"Annual TXx ({first_year})")

plt.tight_layout()

plt.savefig(
    FIG_DIR / f"TXx_{first_year}.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
year = 2050

plt.figure(figsize=(8,6))

txx.sel(year=year).plot(
    cmap="hot",
    robust=True
)

plt.title(f"Annual TXx ({year})")

plt.tight_layout()

plt.show()

-- TXx VUN

In [ ]:
txx_mean = txx.mean(dim=["lat", "lon"])

plt.figure(figsize=(12,5))

txx_mean.plot(marker="o")

plt.grid(True)

plt.title("Annual Mean TXx over Vanuatu")

plt.xlabel("Year")

plt.ylabel(f"TXx ({txx.attrs['units']})")

plt.show()